In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.ensemble import RandomForestClassifier

import xgboost as xgb

# ============================================================
# data loading
# ============================================================

snapshot_id = "20260515_190346_3400a811"
DATA_PATH = f"./data/raw/{snapshot_id}.parquet"
df = pd.read_parquet(DATA_PATH)

# ============================================================
# data cleaning
# ============================================================

# type conversion
df["start_time"] = pd.to_datetime(df["start_time"])
df["end_time"] = pd.to_datetime(df["end_time"])
df["duration_minutes"] = pd.to_numeric(df["duration"])/60
df["Power_kW"] = pd.to_numeric(df["PricePerKWh"])
df["consumption_kwh"] = pd.to_numeric(df["consumption_kwh"])
df["connector_id"] = df["connector_id"].astype(int)
df["latitude"] = pd.to_numeric(df["latitude"])
df["longitude"] = pd.to_numeric(df["longitude"])


# Remove short charging session and low consumption
df = df[
    (df["consumption_kwh"] > 0)
    &
    (df["duration_minutes"] > 1)
]

# Drop missing coordinates/city as we are classifying based on city
df = df.dropna(
    subset=[
        "City",
        "latitude",
        "longitude"
    ]
)

# ============================================================
# feature engineering
# ============================================================

df["hour"] = df["start_time"].dt.hour
df["dayofweek"] = df["start_time"].dt.dayofweek
df["month"] = df["start_time"].dt.month
df["is_weekend"] = (
    df["dayofweek"] >= 5
).astype(int)


# Used city for clustering
df["cluster"] = df["City"]

# Use connector_id approximates actual charging slots.
connectors_per_cluster = (
    df.groupby("cluster")["connector_id"]
      .nunique()
      .reset_index(name="num_connectors")
)
connectors_per_cluster

,cluster,num_connectors
0,Aberdeen,3
1,Aberdeen,1
2,Aberdeenshire,3
3,Aberdour,2
4,Aberlour,3
...,...,...
302,Wishaw,3
303,alva,2
304,lerwick,2
305,st andrews,2


In [5]:
df["hour"].head(20)

1     14
2     14
3     14
4     14
8     14
9     14
10    13
11    13
13    14
14    14
15    14
16    14
17    14
18    14
27    15
28    15
29    15
46     1
47     1
48     1
Name: hour, dtype: int32

In [6]:
df["is_weekend"].head()

1    0
2    0
3    0
4    0
8    0
Name: is_weekend, dtype: int64

In [3]:
df["City"].unique()

<ArrowStringArray>
[        'Lanark',     'Motherwell',         'Dundee',       'Larkhall',
         'Elgin ',  'Auchtermuchty',  'East Kilbride',      'Edinburgh',
        'Glasgow',        'Cumnock',
 ...
         'Rothes',    'Carnoustie ',  'Isle of lewis', 'Broughty Ferry',
       'Creetown',       'Sandwick',       'Straiton',   'Isle of Mull',
  'Bridge of Don',     'Baltasound']
Length: 307, dtype: str